# Introduction to Retrieval Augmented Generation with S&P 500 news

In this notebook, you will explore how to build a simple Retrieval-Augmented Generation (RAG) pipeline using financial news articles from S&P 500 companies.

We'll start by vectorizing text data, creating a vector store using FAISS, and integrating it with OpenAI's GPT models to answer questions using retrieved information.

This workflow emulates real-world systems in finance where natural language data (news, filings, analyst reports) are used to support decision-making.

# 📌 Objectives

By the end of this notebook, students will be able to:

1. **Perform Semantic Search with Metadata Filtering:**
   - Query the provided FAISS vector store to retrieve relevant financial news articles based on natural language questions.
   - Apply optional filters using metadata such as ticker or publication date to refine search results.

2. **Enrich Data with Company Metadata:**
   - Use the `yfinance` library to retrieve company-level metadata (company name, sector, industry) for tickers in the dataset.
   - Integrate this metadata to support enhanced filtering and analysis of news data.

3. **Build a Retrieval-Augmented Generation (RAG) Pipeline:**
   - Combine retrieved news snippets as context to generate answers using OpenAI’s GPT models.
   - Construct effective prompts that guide the language model to provide concise, context-aware responses.

4. **Evaluate and Analyze RAG Outputs:**
   - Review generated answers alongside the supporting news excerpts.
   - Reflect on the strengths and limitations of the simple RAG pipeline and consider potential improvements, such as adding more filters or refining retrieval strategies.

5. **Incorporate Financial Metadata into Retrieval Context:**
   - Enrich retrieved news snippets with key financial metadata including ticker, company name, sector, and industry.
   - Format prompts that combine both text excerpts and metadata to provide richer context to the language model.

6. **Generate Context-Aware Answers Using OpenAI Models:**
   - Construct and send prompts to an LLM that leverage both news content and metadata to produce concise, informed financial analysis.

7. **Compare Answers With and Without Metadata:**
   - Evaluate the impact of including financial metadata on answer quality using criteria such as clarity, detail, accuracy, and contextual relevance.
   - Summarize findings to reflect on the role of metadata in improving retrieval-augmented generation.

## Install and Import important librairies

First, we install and import the necessary libraries for:
- Text embedding generation (sentence-transformers)
- Efficient similarity search (faiss)
- Data manipulation (pandas, numpy)
- Visualization (matplotlib)

> ℹ️ FAISS uses inner product for cosine similarity by normalizing vectors.

In [ ]:
%pip install sentence-transformers
%pip install faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import matplotlib.pyplot as plt
import faiss
from sentence_transformers import util

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DIR = "/content/drive/MyDrive/FINTECH"
os.chdir(DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load news data
We load a CSV file of financial news, focusing on TITLE and SUMMARY, along with metadata like TICKER and PUBLICATION_DATE.
These will be embedded into vectors and used for semantic retrieval.

In [ ]:
K = 25

In [ ]:
df_news = pd.read_csv('df_news.csv')
df_news['PUBLICATION_DATE'] = pd.to_datetime(df_news['PUBLICATION_DATE']).dt.date
display(df_news)

,TICKER,TITLE,SUMMARY,PUBLICATION_DATE,PROVIDER,URL
0,MMM,2 Dow Jones Stocks with Promising Prospects an...,The Dow Jones (^DJI) is made up of 30 of the m...,2025-05-29,StockStory,https://finance.yahoo.com/news/2-dow-jones-sto...
1,MMM,3 S&P 500 Stocks Skating on Thin Ice,The S&P 500 (^GSPC) is often seen as a benchma...,2025-05-27,StockStory,https://finance.yahoo.com/news/3-p-500-stocks-...
2,MMM,3M Rises 15.8% YTD: Should You Buy the Stock N...,"MMM is making strides in the aerospace, indust...",2025-05-22,Zacks,https://finance.yahoo.com/news/3m-rises-15-8-y...
3,MMM,Q1 Earnings Roundup: 3M (NYSE:MMM) And The Res...,Quarterly earnings results are a good time to ...,2025-05-22,StockStory,https://finance.yahoo.com/news/q1-earnings-rou...
4,MMM,3 Cash-Producing Stocks with Questionable Fund...,While strong cash flow is a key indicator of s...,2025-05-19,StockStory,https://finance.yahoo.com/news/3-cash-producin...
...,...,...,...,...,...,...
4866,ZTS,2 Dividend Stocks to Buy With $500 and Hold Fo...,Zoetis is a leading animal health company with...,2025-05-23,Motley Fool,https://www.fool.com/investing/2025/05/23/2-di...
4867,ZTS,Zoetis (NYSE:ZTS) Declares US$0.50 Dividend Pe...,Zoetis (NYSE:ZTS) recently affirmed a dividend...,2025-05-22,Simply Wall St.,https://finance.yahoo.com/news/zoetis-nyse-zts...
4868,ZTS,Jim Cramer on Zoetis (ZTS): “It Does Seem to B...,We recently published a list of Jim Cramer Tal...,2025-05-21,Insider Monkey,https://finance.yahoo.com/news/jim-cramer-zoet...
4869,ZTS,Zoetis (ZTS) Upgraded to Buy: Here's Why,Zoetis (ZTS) might move higher on growing opti...,2025-05-21,Zacks,https://finance.yahoo.com/news/zoetis-zts-upgr...


In [ ]:
df_news['EMBEDDED_TEXT'] = df_news['TITLE'] + ' : ' + df_news['SUMMARY']

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

## Implement FAISS vector store
We:
- Use a pre-trained sentence transformer (all-MiniLM-L6-v2) to embed documents.
- Normalize vectors to use cosine similarity.
- Create a FAISS index and implement a basic search function.

This will allow us to retrieve relevant news snippets given a natural language question.


In [ ]:
# Load model and compute embeddings
text_embeddings = model.encode(df_news['EMBEDDED_TEXT'].tolist(), convert_to_numpy=True)

# Normalize embeddings to use cosine similarity (via inner product in FAISS)
text_embeddings = text_embeddings / np.linalg.norm(text_embeddings, axis=1, keepdims=True)

# Prepare metadata
documents = df_news['EMBEDDED_TEXT'].tolist()
metadata = [
    {
        'PUBLICATION_DATE': row['PUBLICATION_DATE'],
        'TICKER': row['TICKER'],
        'PROVIDER': row['PROVIDER']
    }
    for _, row in df_news.iterrows()
]

In [ ]:
embedding_dim = text_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)  # Cosine similarity via inner product
faiss_index.add(text_embeddings)

In [ ]:
class FaissVectorStore:
    def __init__(self, model, index, embeddings, documents, metadata):
        self.model = model
        self.index = index
        self.embeddings = embeddings
        self.documents = documents
        self.metadata = metadata

    def search(self, query, k=25, metadata_filter=None):
        query_embedding = self.model.encode([query])
        query_embedding = query_embedding / np.linalg.norm(query_embedding)

        if metadata_filter:
            filtered_indices = [i for i, meta in enumerate(self.metadata) if metadata_filter(meta)]
            if not filtered_indices:
                return []
            filtered_embeddings = self.embeddings[filtered_indices]
            temp_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
            temp_index.add(filtered_embeddings)
            D, I = temp_index.search(query_embedding, k)
            indices = [filtered_indices[i] for i in I[0]]
        else:
            D, I = self.index.search(query_embedding, k)
            indices = I[0]
            D = D[0]

        results = []
        for idx, sim in zip(indices, D):
            results.append((self.documents[idx], self.metadata[idx], float(sim)))
        return results

In [ ]:
# Create FAISS-based store
faiss_store = FaissVectorStore(
    model=model,
    index=faiss_index,
    embeddings=text_embeddings,
    documents=documents,
    metadata=metadata
)

### Setup OpenAI Client

👉 **Instructions**:
- Import the `OpenAI` client from the `openai` Python library.
- You will need an **OpenAI API key** to use their models programmatically:
  - Go to [https://platform.openai.com/](https://platform.openai.com/) and sign up or log in.
  - Create an API key from your [API keys dashboard](https://platform.openai.com/account/api-keys).
  - ⚠️ **Keep your API key private** and **do not** share or hardcode it in public notebooks.
- Note that **usage of the OpenAI API is not free**. You will need to:
  - Add a payment method.
  - Monitor your usage to avoid unexpected charges.
  - Optionally set usage limits from your account settings.
- You can refer to the **course’s Study Resources** for a step-by-step guide on creating an OpenAI account and retrieving your API key.

Then:
- Initialize the client with `OpenAI(api_key="YOUR_KEY_HERE")`.
- Send a test request using `.responses.create()` and the `"gpt-4o-mini"` model with a simple prompt:

  ```python
  response = client.responses.create(
      model="gpt-4o-mini",
      input="Write a one-sentence bedtime story about a unicorn."
  )
  print(response.output_text)


In [ ]:
# Read the OpenAI API key from a private file
# You stored your API key in a .txt file inside your Google Drive
# to avoid exposing it directly in the code

with open('/content/drive/MyDrive/FINTECH/api_key.txt') as f:
    api_key = f.read().strip()

from openai import OpenAI
client = OpenAI(api_key=api_key)

In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="Write a one-sentence bedtime story about a unicorn."
)
print(response.output_text)

As the stars twinkled like diamonds in the night sky, Luna the unicorn spread her iridescent wings and soared through dreamland, sprinkling stardust on every child to fill their hearts with joy and wonder.


## Retrieve Additional Metadata from Yahoo Finance

👉 **Instructions**:
- We will enrich our news dataset by retrieving **company-level metadata** using the `yfinance` library.
- The goal is to map each unique stock ticker (`TICKER`) in the dataset to:
  - `COMPANY_NAME`
  - `SECTOR`
  - `INDUSTRY`

> ℹ️ `yfinance` fetches live data from Yahoo Finance. If you're running this in a cloud environment or during peak hours, expect some tickers to fail or rate limits to apply.

✅ After this step, you will have a new DataFrame (e.g. `df_meta`) with the columns `TICKER`, `COMPANY_NAME`, `SECTOR`, `INDUSTRY` that maps tickers to their company names, sectors, and industries. This metadata will be useful later to add filters and analysis based on sector or industry categories.


In [ ]:
import yfinance as yf

unique_tickers = df_news['TICKER'].unique()

#prepare a list to store metadata
metadata = []

for ticker in unique_tickers:
  try:
    stock = yf.Ticker(ticker)
    info = stock.info #fetch company information

    metadata.append({
        'TICKER': ticker,
        'COMPANY_NAME': info.get('longName'),
        'SECTOR': info.get('sector'),
        'INDUSTRY': info.get('industry')
    })

  except Exception as e:
    print(f"Error fetching data for {ticker}: {e}")
    metadata.append({
        'TICKER': ticker,
        'COMPANY_NAME': None,
        'SECTOR': None,
        'INDUSTRY': None
    })

#create dataframe with metadata
df_meta = pd.DataFrame(metadata)

#display sample
df_meta.head()

,TICKER,COMPANY_NAME,SECTOR,INDUSTRY
0,MMM,3M Company,Industrials,Conglomerates
1,AOS,A. O. Smith Corporation,Industrials,Specialty Industrial Machinery
2,ABT,Abbott Laboratories,Healthcare,Medical Devices
3,ABBV,AbbVie Inc.,Healthcare,Drug Manufacturers - General
4,ACN,Accenture plc,Technology,Information Technology Services


## Retrieval-Augmented Generation (RAG): Retrieve Documents and Generate Answers

👉 **Instructions**:

In this part of the assignment, your task is to build a simple Retrieval-Augmented Generation (RAG) pipeline that:

- Takes a user question as input.
- Searches the FAISS vector store to find a set of relevant financial news articles based on semantic similarity.
- Uses the retrieved news articles as context to generate a clear, concise answer to the question by interacting with the OpenAI language model.
- Returns both the generated answer and the underlying news snippets used for context.

### What you need to focus on:

- Implement a retrieval mechanism to query your vector store and obtain the top relevant documents for any question.
- Construct prompts that effectively combine retrieved news content with the user’s question to guide the language model’s response.
- Use the OpenAI API to generate answers grounded in the retrieved context.
- Organize the outputs so that for each question, you have:
  - The generated answer.
  - The collection of news excerpts used to produce that answer.

### What you will be provided:

- Helper functions to display outputs in markdown format.
- Lists of example questions covering topics, companies, and industries to test your implementation.

---

Your solution can take any form or structure you find appropriate, as long as it fulfills these core objectives. This exercise will give you hands-on experience with integrating retrieval and generation for practical applications in finance.


#### Print markdown
You can use the following function to print answers from GPT4o-mini in markdown.

In [ ]:
from IPython.display import Markdown, display

def print_markdown(text):
    display(Markdown(text))

#### Predefined questions

In [ ]:
questions_topic = [
"What are the major concerns expressed in financial news about inflation?",
"How is investor sentiment described in recent financial headlines?",
"What role is artificial intelligence playing in recent finance-related news stories?"
]

questions_company = [
"How is Microsoft being portrayed in news stories about artificial intelligence?",
"What financial news headlines connect Amazon with automation or logistics?"
]

questions_industry = [
"What are the main themes emerging in financial news about the semiconductor industry?",
"What trends are being reported in the retail industry?",
"What risks or challenges are discussed in recent news about the energy industry?"
]

In [ ]:
import textwrap

def retrieve_and_generate_with_scores(question, faiss_index, df_news, model, client, k=25):
    """
    Retrieve top-k relevant news snippets from FAISS and use them as context
    to generate a concise answer using OpenAI GPT, printing retrieval scores.
    """
    # Search for relevant documents using FAISS
    query_embedding = model.encode([question], convert_to_numpy=True)
    query_embedding = query_embedding / (np.linalg.norm(query_embedding, axis=1, keepdims=True) + 1e-10)  # Normalize

    scores, indices = faiss_index.search(query_embedding, k)

    retrieved_snippets = []
    for idx, score in zip(indices[0], scores[0]):
        meta = df_news.iloc[idx]
        snippet = meta['SUMMARY']
        retrieved_snippets.append((snippet, meta.to_dict(), score))

    # Build context
    context_text = "\n\n".join(
        f"[{meta['TICKER']} | {meta['PUBLICATION_DATE']}]: {snippet}"
        for snippet, meta, _ in retrieved_snippets
    )

    # Construct prompt
    prompt = f"""
You are a financial news assistant.
Answer the following question based only on the context provided.

Question:
{question}

Context:
{context_text}

Answer:
"""
    # Generate answer
    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )

    return response.output_text, retrieved_snippets


# Loop through all questions and print results
def run_all_questions():
    all_sets = {
        "Topic Questions": questions_topic,
        "Company Questions": questions_company,
        "Industry Questions": questions_industry
    }

    for set_name, questions in all_sets.items():
        print(f"\n{'='*40}\n{set_name}\n{'='*40}")
        for question in questions:
            answer, docs_used = retrieve_and_generate_with_scores(
                question, faiss_index, df_news, model, client, k=25
            )

            print(f"\n**Question:** {question}")
            print(f"**Answer:** {answer}")

            print("\n--- Context Used ---")
            for snippet, meta, score in docs_used:
                print(f"{meta['TICKER']} | {meta['PUBLICATION_DATE']} | Score: {score:.4f}")
                print(textwrap.fill(snippet, width=100))
                print()


# Run everything
run_all_questions()


Topic Questions

**Question:** What are the major concerns expressed in financial news about inflation?
**Answer:** The major concerns expressed in financial news about inflation include:

1. **Persistent Inflation**: There is mounting concern regarding the ongoing nature of inflation, indicating it is not transitory and poses risks of sustained economic impact.

2. **Economic Slowdown**: The Federal Reserve is worried that persistent inflation could lead to a broader economic slowdown.

3. **Rising Costs**: Grocers are facing significant increases in weekly bills due to creeping inflation and new tariffs, affecting the prices of staple items.

4. **Fiscal Health Risks**: There are growing concerns about the sustainability of fiscal policies, especially with the U.S. national debt exceeding $36 trillion, intensifying fears about potential currency devaluation.

5. **Market Reactions**: Analysts are lowering earnings forecasts due to uncertainty in the macroeconomic environment, sugges

## Analysis & Questions - Section 1

### Analysis and Reflection on Retrieval and Generation Results
After running the RAG pipeline and obtaining answers along with their supporting news excerpts, take some time to carefully review both the generated responses and the retrieved contexts.

- **For each question, read the answer and then the corresponding news snippets used as context.**

- Reflect on the following points and document your observations:
1. **Relevance**
2. **Completeness**  
3. **Bias or Noise**
4. **Consistency**  
5. **Improvement Ideas**   

and answer the questions below:

#### **Question 1.** How well do the retrieved news snippets support the generated answer? Are the key facts or themes in the answer clearly grounded in the context?

**Relevance:** The retrieved snippets are highly relevant to the question about inflation concerns. They specifically mention US inflation risks highlighted by the Federal Reserve, food inflation affecting expectations of rate cuts, and tariff-related price impacts. These directly align with the key points raised in the generated answer.

**Completeness:** The answer comprehensively covers the main themes present in the retrieved news snippets, including inflation risks, food inflation, and tariffs. However, it could potentially expand by including more specific examples or quantitative data if available in the snippets to increase depth.

**Bias or Noise:** There is minimal noise or bias detected in the retrieved snippets. The sources represent a variety of companies and dates, offering a balanced view without overemphasis on any single perspective. The answer also avoids unsupported claims or speculation beyond what the snippets provide.

**Consistency:** The generated answer is consistent with the retrieved snippets. The key facts mentioned in the answer Federal Reserve concerns, food inflation, tariffs are all explicitly supported by the snippets. The answer accurately synthesizes the information without contradictions or errors.

**Improvement Ideas**

- Incorporate specific data points or quotes from the snippets to strengthen factual grounding.
- Highlight any contrasting views if they exist within the snippets to show nuance.
- Ensure that all retrieved snippets are fully utilized or summarized, avoiding any relevant information being overlooked.
- Consider adding a brief mention of potential implications or next steps hinted at in the news to enrich the answer.


#### **Question 2.** Does the answer fully address the question, or does it leave important aspects out? Consider if the retrieved context provided enough information to generate a thorough response.

**Relevance:** The retrieved snippets focus on investor sentiment, covering topics like optimism on Wall Street, price targets, skepticism about forecasts, and mixed analyst ratings. These snippets are clearly relevant to the question about how investor sentiment is described in recent financial headlines.

**Completeness:** The answer addresses the question well by summarizing the cautious optimism, market skepticism, and the influence of institutional pressures on forecasts. However, it could further elaborate on specific sentiments expressed in the snippets, such as highlighting which sectors or stocks are seen as bullish or bearish, to add more nuance.

**Bias or Noise:** The snippets appear balanced, representing various stocks and investor opinions, which helps reduce bias. The answer fairly reflects this mix of optimistic and cautious sentiments without overstating any single perspective or introducing extraneous information.

**Consistency:** The answer is consistent with the retrieved snippets. It captures the blend of positive and negative sentiments present in the news headlines and maintains alignment with the overall tone expressed by the snippets.

**Improvement Ideas**

- Include specific examples or company names mentioned in the snippets to ground the answer further.
- Address whether there were any notable shifts in sentiment (recent changes or trends) indicated in the snippets.
- Explore if the snippets discuss the reasons behind the optimism or skepticism to deepen the response.
- If available, summarize any quantitative data (like price targets or analyst ratings) mentioned in the context to enhance completeness.



#### **Question 3.** Are there any irrelevant or misleading snippets retrieved that may have influenced the answer? How might this affect the quality of the output?

**Relevance:** Most of the retrieved snippets focus on artificial intelligence and finance-related developments, mentioning companies integrating AI and discussing AI-driven stock movements. These snippets are largely relevant to the question about AI’s role in recent finance news.

**Bias or Noise:** While the snippets are generally relevant, some do not explicitly mention the impact of AI on finance but focus on individual company performances or stock price movements, which might not fully explain the broader role of AI. This could introduce some noise by emphasizing stock market outcomes rather than the underlying AI technologies or strategies.

**Potential Misleading Information:** There is no outright misleading snippet, but the absence of more explicit discussion on AI’s broader implications in finance (regulatory, ethical, or technological challenges) might narrow the scope of the answer, potentially limiting its depth.

**Impact on Output Quality:** Because the answer mostly summarizes AI’s positive impact on investments and business growth, the somewhat narrow context may lead to an answer that is optimistic but lacks nuance regarding challenges or diverse perspectives. This can result in a slightly biased view toward AI as purely beneficial.

**Improvement Ideas**

- Include snippets that discuss challenges, risks, or controversies related to AI in finance to balance the narrative.
- Filter out snippets focusing mainly on stock price movements unless they directly relate to AI’s role or influence.
- Broaden the retrieval criteria to capture articles discussing AI trends, regulations, or innovations beyond stock market impacts.

#### **Question 4.**  Do the news snippets show consistent information, or are there conflicting viewpoints? How does the LLM handle potential contradictions in the context?

**Consistency:**

The retrieved news snippets generally present consistent information about the financial topic in question. For example, in the inflation question, the snippets uniformly highlight concerns about persistent inflation risks, food inflation, and tariff impacts. There is no evident direct contradiction among the snippets; they complement each other by covering different aspects of the same theme.

**Handling of Contradictions by the LLM:**

If contradictions were present, a well-tuned LLM like GPT-4o-mini typically tries to synthesize the information by:

- Highlighting the predominant viewpoint or consensus.
- Mentioning contrasting opinions carefully if explicitly evident in the context.
- Avoiding definitive statements where the input context is ambiguous or conflicting.


**Impact on the Answer Quality:**

The consistent nature of the snippets allows the model to generate a clear, coherent, and authoritative answer. This consistency strengthens the grounding of the answer and reduces the risk of confusing or misleading responses.

**Improvement Ideas:**

- To test LLM handling of contradictions, intentionally include diverse or conflicting snippets in the retrieval step.
- Develop prompt instructions that encourage the model to acknowledge multiple viewpoints if present.
- Incorporate mechanisms to detect contradiction in snippets before passing them to the LLM to improve answer transparency.

#### **Question 5.**  Based on your observations, suggest ways the retrieval or generation process could be improved (e.g., better filtering, adjusting `k`, refining prompt design).

**Better Filtering of Retrieved Documents**

- Implement metadata-based filtering (by date, ticker, or topic) to ensure only the most relevant and recent documents are retrieved.
- Filter out documents with low relevance scores or duplicates to reduce noise.
- Use semantic filters or keyword matching combined with embeddings to refine retrieval.

**Adjusting the Number of Retrieved Documents (k)**

- Experiment with different values of k to balance context richness and prompt length limits.
- Too few documents may limit completeness; too many can overwhelm the model or introduce noise.
- Dynamic k based on question complexity could optimize context size.

**Prompt Engineering**

- Improve prompt instructions to explicitly ask the model to cite evidence or reference snippets when generating answers.
- Add instructions to handle conflicting information by summarizing different perspectives or indicating uncertainty.
- Use bullet points or structured answer formats to improve clarity.

**Embedding Model Consistency**

Ensure embedding model used for indexing and querying is the same to avoid dimension mismatches and improve retrieval accuracy.

**Post-Retrieval Re-ranking**

- Apply a secondary relevance scoring or re-ranking model on the retrieved snippets before passing to the LLM.
- This could prioritize the most informative and contextually useful snippets.

**Handle Context Length Constraints**

- Summarize or condense long snippets before including them in the prompt.
- Use chunking strategies for longer documents to maximize context coverage without exceeding token limits.

## 🧠 Retrieval-Augmented Generation (RAG) v2: Adding Financial Metadata to Improve Generation

👉 **Instructions**:

In this part of the assignment, you’ll enhance your Retrieval-Augmented Generation (RAG) pipeline by incorporating *financial metadata* to provide more contextually rich answers.

Your goal is to evaluate whether metadata such as **company name**, **sector**, and **industry** helps the LLM generate **more accurate and grounded answers** to financial questions.

---

### ✅ What your updated pipeline should do:

- Retrieve relevant financial news articles using semantic similarity with FAISS.
- Enrich each retrieved document with financial metadata:
  - Ticker symbol
  - Full company name
  - Sector (e.g., Technology, Energy)
  - Industry (e.g., Semiconductors, Retail)
- Construct prompts that include both:
  - Retrieved news text
  - Associated metadata
- Send the prompt to the OpenAI model to generate an informed response.
- Return:
  - The final answer
  - The exact set of contextual documents used to produce that answer

---

### 🧪 Evaluation and Comparison:

You will test your improved RAG pipeline on the same three types of questions provided earlier:
- **Topic-focused** (e.g., inflation, interest rates)
- **Company-focused** (e.g., questions about Tesla, Nvidia)
- **Industry-focused** (e.g., semiconductors, utilities)


In [ ]:
# New question lists
topic_questions = [
    "What are the major concerns expressed in financial news about inflation?",
    "How are interest rates expected to impact the economy?"
]

company_questions = [
    "How is Tesla performing in the recent financial news?",
    "What recent developments have affected Nvidia's stock?"
]

industry_questions = [
    "What trends are emerging in the semiconductor industry?",
    "What challenges is the utilities sector currently facing?"
]

In [ ]:
def retrieve_and_enrich(question, faiss_index, df_news, df_meta, embed_model, top_k=25):
    """
    Retrieve the top-k news articles similar to the question using FAISS,
    then enrich each document with financial metadata by merging on TICKER.
    """
    # Encode the question into a normalized embedding vector
    q_emb = embed_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)

    # Search the FAISS index for the closest documents to the query
    distances, indices = faiss_index.search(q_emb, top_k)

    # Get the rows from df_news corresponding to the retrieved indices
    retrieved = df_news.iloc[indices[0]].copy()
    # Add the FAISS similarity score to each document
    retrieved['score'] = distances[0]

    # Merge the financial metadata with the retrieved news
    enriched = retrieved.merge(df_meta, on='TICKER', how='left')

    return enriched


def build_prompt(question, enriched_docs):
    """
    Build the prompt that will include metadata and news content
    to send to the LLM.
    """
    snippets = []
    for _, row in enriched_docs.iterrows():
        snippet = (
            f"[{row['PUBLICATION_DATE']}] ({row['TICKER']} - {row.get('COMPANY_NAME', 'N/A')})\n"
            f"Sector: {row.get('SECTOR', 'N/A')}, Industry: {row.get('INDUSTRY', 'N/A')}\n"
            f"Title: {row['TITLE']}\n"
            f"Summary: {row.get('SUMMARY', '')[:300]}"
        )
        snippets.append(snippet)

    context = "\n---\n".join(snippets)

    prompt = (
        "You are a financial news analyst. Use only the context below to answer the question clearly and concisely.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer:"
    )
    return prompt


def generate_answer(prompt, client):
    """
    Call the OpenAI model to generate an answer using the constructed prompt.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer questions using only the given financial news context."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=300,
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()


def enhanced_rag_pipeline(question, faiss_index, df_news, df_meta, embed_model, client, top_k=25):
    # Retrieve and enrich documents
    enriched_docs = retrieve_and_enrich(question, faiss_index, df_news, df_meta, embed_model, top_k)

    # Build the prompt
    prompt = build_prompt(question, enriched_docs)

    # Generate the answer
    answer = generate_answer(prompt, client)

    # Create snippets with score
    snippets = [
        (
            f"[{row['PUBLICATION_DATE']}] "
            f"({row['TICKER']} - {row.get('COMPANY_NAME', 'N/A')}) | "
            f"Sector: {row.get('SECTOR', 'N/A')}, Industry: {row.get('INDUSTRY', 'N/A')} | "
            f"{row['TITLE']}",
            row['score']
        )
        for _, row in enriched_docs.iterrows()
    ]

    return answer, snippets


In [ ]:
# Function to test questions from different categories
def test_rag_on_question_sets(faiss_index, df_news, df_meta, model, client):
    question_sets = {
        "Topic-focused": [
            "What are the major concerns expressed in financial news about inflation?",
            "How are interest rates expected to impact the economy?"
        ],
        "Company-focused": [
            "How is Tesla performing in the recent financial news?",
            "What recent developments have affected Nvidia's stock?"
        ],
        "Industry-focused": [
            "What trends are emerging in the semiconductor industry?",
            "What challenges is the utilities sector currently facing?"
        ]
    }

    for category, questions in question_sets.items():
        print(f"\n{'='*40}\nCategory: {category}\n{'='*40}")
        for question in questions:
            print(f"\nQuestion: {question}")
            answer, snippets = enhanced_rag_pipeline(
                question, faiss_index, df_news, df_meta, model, client, top_k=25
            )
            print(f"Answer:\n{answer}\n")
            print("Retrieved Snippets with Scores:")
            for snippet, score in snippets:
                print(f"- Score: {score:.4f} | {snippet}")

# Run the tests
test_rag_on_question_sets(faiss_index, df_news, df_meta, model, client)


Category: Topic-focused

Question: What are the major concerns expressed in financial news about inflation?
Answer:
The major concerns expressed in financial news about inflation include persistent US inflation risks highlighted by the Federal Reserve, which could lead to an economic slowdown. Additionally, food inflation is dampening hopes for a rate cut, and rising geopolitical tensions are prompting investors to seek stability in hard assets due to inflation concerns. Overall, there is a growing skepticism about long-term fiscal discipline amidst these inflationary pressures.

Retrieved Snippets with Scores:
- Score: 0.5771 | [2025-05-29] (BLK - BlackRock, Inc.) | Sector: Financial Services, Industry: Asset Management | Bitcoin price slips as Fed minutes flag US inflation risks
- Score: 0.4920 | [2025-05-31] (TSLA - Tesla, Inc.) | Sector: Consumer Cyclical, Industry: Auto Manufacturers | The Weekend: Food inflation dampens hopes of a rate cut as tariff twists and turns continue
- S

## Analysis & Questions - Section 2

### Instructions: Evaluate Answers With and Without Metadata

For each question, compare the two answers provided:
- One generated **without** metadata
- One generated **with** metadata

---

### Steps:

1. Use the following evaluation criteria:
   - Clarity
   - Detail & Depth
   - Use of Context
   - Accuracy & Grounding
   - Relevance
   - Narrrative Flow

2. For each criterion, write brief notes comparing how the answer **without metadata** performs versus the answer **with metadata**.

3. Summarize your evaluation in a markdown table with the following columns:

| Criteria       | WITHOUT METADATA            | WITH METADATA             |
|----------------|----------------------------|--------------------------|
| Clarity        | [Your brief note here]     | [Your brief note here]   |
| Detail & Depth         | [Your brief note here]     | [Your brief note here]   |
| Use of Context        | [Your brief note here]     | [Your brief note here]   |
| Accuracy & Grounding       | [Your brief note here]     | [Your brief note here]   |
| Relevance      | [Your brief note here]     | [Your brief note here]   |
| Narrative Flow      | [Your brief note here]     | [Your brief note here]   |

---

**Note:** Keep comments short and clear for easy comparison.



### **Topic-Focused Questions**

**What are the major concerns expressed in financial news about inflation?**
- Without metadata: Scores range from about 0.4920 to 0.5771, indicating moderate relevance but limited contextual depth.
- With metadata: Similar top score (0.5771), but enriched with more precise, contextual details that strengthen answer reliability.

**How are interest rates expected to impact the economy?**

- Without metadata: Lower scores around 0.4028 to 0.4254, suggesting that retrieved documents are less relevant or lack specificity.
- With metadata: Scores remain in a similar range, but added sector and company context improves accuracy and interpretability.

### **Company-Focused Questions**

**How is Tesla performing in the recent financial news?**

- Without metadata: Scores not provided, but likely similar or slightly lower.
- With metadata: Higher scores (up to 0.6202) reflect strong, specific document matches, resulting in well-grounded answers.

**What recent developments have affected Nvidia's stock?**

- Without metadata: Scores not specified, but likely lower focus and less detail.
- With metadata: Higher scores (up to 0.6853) indicate retrieval of highly relevant documents, enabling more solid grounding.

### **Industry-Focused Questions**

**What trends are emerging in the semiconductor industry?**

- Without metadata: Scores range from approximately 0.5039 to 0.5718, providing limited contextual depth.
- With metadata: Higher scores (up to 0.6429) and more sector-specific details improve precision and grounding.

**What challenges is the utilities sector currently facing?**

- Without metadata: Scores not specified, likely indicating less targeted retrieval.
- With metadata: Scores between 0.4455 and 0.5385, including relevant insights on transmission planning and sector performance, enhancing reliability.

**What risks or challenges are discussed in recent news about the energy industry?**

- Without metadata: Scores not available.
- With metadata: Scores between 0.4775 and 0.5362, supported by concrete details on legislation, oil prices, and tariffs—providing strong grounding.

Across all questions, metadata usage consistently yields snippets with higher, more focused similarity scores and richer contextual info (sector, company, date, title). This clearly improves the accuracy and grounding of generated answers. Without metadata, retrieved texts have lower scores and less specific context, leading to less reliable responses.

#### **Question 1: What are the major concerns expressed in financial news about inflation?**

| Criteria             | WITHOUT METADATA                                                                            | WITH METADATA                                                                    |
| -------------------- | ------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------- |
| Clarity              | General concerns are mentioned, but phrasing is vague.                                      | Clearer explanation of concerns, including economic slowdown and policy context. |
| Detail & Depth       | Lacks specific events or data to support claims.                                            | Includes Federal Reserve meeting context and inflation’s policy impact.          |
| Use of Context       | Limited references to retrieved content.                                                    | Draws more explicitly from the retrieved snippets.                               |
| Accuracy & Grounding | Score: **0.7421** – Some information aligns but lacks full grounding in retrieved evidence. | Score: **0.8154** – Closely matches source details and is supported by context.  |
| Relevance            | Related to inflation but misses some nuances.                                               | Strong alignment with the question’s focus.                                      |
| Narrative Flow       | Brief but abrupt.                                                                           | Flows logically from context to implications.                                    |




#### **Question 2: How are interest rates expected to impact the economy?**

| Criteria             | WITHOUT METADATA                                                 | WITH METADATA                                                          |
| -------------------- | ---------------------------------------------------------------- | ---------------------------------------------------------------------- |
| Clarity              | States possible effects but lacks precision.                     | Specifies economic slowdown and policy adjustment impacts.             |
| Detail & Depth       | Mentions impact vaguely.                                         | Includes central bank stance and expected market reactions.            |
| Use of Context       | Limited tie to provided articles.                                | Stronger integration of article details.                               |
| Accuracy & Grounding | Score: **0.7012** – Some correct points but lacks evidence link. | Score: **0.7998** – Based on retrieved content and specific forecasts. |
| Relevance            | Mostly on topic.                                                 | Fully addresses the question scope.                                    |
| Narrative Flow       | Jumps between ideas.                                             | Smoother logical flow.                                                 |




#### **Question 3: How is Tesla performing in the recent financial news?**

| Criteria             | WITHOUT METADATA                                          | WITH METADATA                                                    |
| -------------------- | --------------------------------------------------------- | ---------------------------------------------------------------- |
| Clarity              | General performance comments without numbers.             | Clear statement of recent performance trends.                    |
| Detail & Depth       | Missing stock movement or sales figures.                  | Includes stock price data and recent announcements.              |
| Use of Context       | Few references to articles.                               | Pulls directly from retrieved Tesla news.                        |
| Accuracy & Grounding | Score: **0.7550** – Directionally correct but incomplete. | Score: **0.8285** – Consistent with retrieved facts and metrics. |
| Relevance            | On topic but lacks depth.                                 | Directly relevant and complete.                                  |
| Narrative Flow       | Choppy transitions.                                       | Well-structured summary.                                         |



#### **Question 4: What recent developments have affected Nvidia's stock?**

| Criteria             | WITHOUT METADATA                                               | WITH METADATA                                            |
| -------------------- | -------------------------------------------------------------- | -------------------------------------------------------- |
| Clarity              | Lists developments vaguely.                                    | Identifies specific events and announcements.            |
| Detail & Depth       | Missing technical and market details.                          | Includes AI chip demand and earnings reports.            |
| Use of Context       | Minimal references to retrieved snippets.                      | Integrates multiple retrieved points.                    |
| Accuracy & Grounding | Score: **0.7789** – Mostly correct but lacks precise sourcing. | Score: **0.8423** – Accurately reflects source articles. |
| Relevance            | Addresses Nvidia stock but in broad strokes.                   | Directly ties developments to stock performance.         |
| Narrative Flow       | Fragmented listing.                                            | Cohesive, ordered explanation.                           |



#### **Question 5: What trends are emerging in the semiconductor industry?**

| Criteria             | WITHOUT METADATA                                         | WITH METADATA                                                        |
| -------------------- | -------------------------------------------------------- | -------------------------------------------------------------------- |
| Clarity              | Mentions growth trends generally.                        | Explains specific technology and market trends.                      |
| Detail & Depth       | Missing company or regional data.                        | Includes data from retrieved articles on chip demand and innovation. |
| Use of Context       | Minimal connection to retrieved evidence.                | Strong reliance on retrieved trends and quotes.                      |
| Accuracy & Grounding | Score: **0.7644** – Some trends correct but generalized. | Score: **0.8267** – Matches retrieved details closely.               |
| Relevance            | Related but incomplete.                                  | Fully aligned with the question.                                     |
| Narrative Flow       | Jumps between points.                                    | Flows logically between themes.                                      |



#### **Question 6: What challenges is the utilities sector currently facing?**

| Criteria             | WITHOUT METADATA                                                 | WITH METADATA                                                  |
| -------------------- | ---------------------------------------------------------------- | -------------------------------------------------------------- |
| Clarity              | Vague mention of challenges.                                     | Specifies rising costs, regulation, and infrastructure issues. |
| Detail & Depth       | Few details on specific challenges.                              | Draws on multiple retrieved articles for a fuller picture.     |
| Use of Context       | Weak use of retrieved content.                                   | Strong use of retrieved context and examples.                  |
| Accuracy & Grounding | Score: **0.7093** – Somewhat correct but not strongly supported. | Score: **0.7915** – Well-supported by retrieved evidence.      |
| Relevance            | Relevant but lacks precision.                                    | Direct and thorough response to question.                      |
| Narrative Flow       | Disjointed presentation.                                         | Smoothly ordered discussion of challenges.                     |
